In [ ]:
# Ensure gradalg is importable when this notebook is opened
# directly (not via pytest). Walks up from the notebook's
# directory to the repo root and prepends it to sys.path if
# gradalg isn't already installed into this kernel.
try:
    import gradalg  # noqa: F401
except ModuleNotFoundError:
    import sys
    from pathlib import Path
    here = Path.cwd().resolve()
    for candidate in (here, *here.parents):
        if (candidate / "gradalg" / "__init__.py").is_file():
            sys.path.insert(0, str(candidate))
            break
    import gradalg  # noqa: F401

# 03 — Poisson Geometri

Bu notebook [03_poisson_geometry.md](03_poisson_geometry.md) markdown'ının çalıştırılabilir sürümüdür. Symplectic manifold üstünde Poisson bracket'inin üç eşdeğer görüşü (derived, Hamiltonian, Koszul) ve `[π, π]_SN = 0` tek koşuluna indirgenmiş Jacobi ispatı.

## Symplectic manifold — (ω, π, ♭, ♯) demet

`SymplecticManifold(ω, bivector=π)` formu, ters-bivector'ı, musical map'leri ve `MusicalCompatibility` aksiyomunu tek objede tutar. Registry'de `ω` 2-form, `π` SN-derecesi 1'lik 2-vektör olarak deklare edilir — `Bivector` yardımcısı bunu otomatik yapar.

In [ ]:
from gradalg import Bivector, Forms, Functions
from gradalg.core.registry import PropertyRegistry
from gradalg.library.symplectic import SymplecticManifold

reg = PropertyRegistry()
(omega,) = Forms("ω", degree=2, registry=reg)
pi = Bivector("π", registry=reg)

M = SymplecticManifold(omega, bivector=pi, name="(M, ω, π)")
print(M)
print('flat:', M.flat)
print('sharp:', M.sharp)
print('compat:', M.compatibility.name)

## `PoissonBracket` — üç eşdeğer görüş

Fonksiyonlar SN-shifted grading'de degree `−1` taşır; `Functions` yardımcısına `degree=-1` kwarg'ı verilir.

In [ ]:
from gradalg.library.poisson import PoissonBracket

f, g, h = Functions("f g h", degree=-1, registry=reg)
poisson = PoissonBracket.from_bivector(pi)
print('bracket:', poisson)

### Görüş 1 — derived bracket

`{f, g}_π = [[f, π]_SN, g]_SN`.

In [ ]:
poisson.expand(f, g, reg)

### Görüş 2 — Hamiltonian vector field

`{f, g}_π = X_f(g)`. Symplectic manifold üstünde bu eşitlik `ι_{X_f} ω + df = 0`'ya denktir; `prove_hamiltonian_equivalence` musical kompatibiliteyi kullanarak beş adımda kapatır.

In [ ]:
from gradalg.display import chain_to_ascii

print('X_f =', poisson.hamiltonian_vf(f))
print('X_f(g) =', poisson.via_hamiltonian(f, g))

chain = M.prove_hamiltonian_equivalence(f, registry=reg)
print('chain length:', len(chain))
print(chain_to_ascii(chain))

### Görüş 3 — Koszul üç-terim formülü

1-formlar üstünde `{α, β}_π = L_{π♯(α)} β − L_{π♯(β)} α − d⟨π♯(α), β⟩`. Klasik Koszul bracket ile derived bracket'in bu operand tipinde *yapısal eşitliği* `prove_koszul_equivalence` ile tek reflexive adımda kayda geçer.

In [ ]:
alpha, beta = Forms("α β", degree=1, registry=reg)
print('koszul expand:', poisson.koszul_expand(alpha, beta, reg))

chain_k = poisson.prove_koszul_equivalence(alpha, beta, registry=reg)
print('koszul chain length:', len(chain_k))
print('rule:', chain_k.steps[0].rule)

## `[π, π]_SN = 0` — tek koşul

Derived Bracket Teoremi, `{·, ·}_π` üstündeki Jacobi'yi tek koşula indirger: `[π, π]_SN = 0`. Üç-giriş reduction zinciri `DerivedBracketTheorem` rule'u ile tek adımda obstruction'a varır; atomik `π` için obstruction opak kalır — Poisson hipotezi devreye girince Jacobi kapanır.

In [ ]:
print('obstruction:', poisson.jacobi_obstruction(reg))
print('condition:', poisson.jacobi_condition(reg))

chain_j = poisson.prove_jacobi_reduction(f, g, h, registry=reg)
print('chain length:', len(chain_j))
print('rule:', chain_j.steps[0].rule)
print('reduces to:', chain_j.steps[0].after)

## Theorem Book — seeded teorem

Kütüphane bu indirgemeyi `poisson_jacobi` altında hazır bir `Theorem` kaydı olarak tutar — downstream kod tek citation ile sonuca bağlanır.

In [ ]:
from gradalg.library import theorem_book

thm = theorem_book.get("poisson_jacobi")
print('statement:', thm.statement)
print('from_axioms:', thm.from_axioms)

## Sonraki adım

Lie algebroid çerçevesi aynı derivation stratejisini bir vector bundle'ın üstünde yaşayan bir bracket'e uygular — [04_lie_algebroid.md](04_lie_algebroid.md).